# Sprint 18b — Generate Test Dataset (512-step, 8Hz, 5 Anomaly Types)

**Cập nhật Sprint 18b:**
- **Kích thước cố định:** Mỗi cycle = **512 steps** (trước đây: biến động 85–600+)
- **Tần số lấy mẫu:** **8 Hz** (125ms/step) — 512 steps = 64 giây
- **Số loại lỗi:** Giảm từ 10 → **5 loại phổ biến nhất**
- **Dữ liệu Train hội tụ:** Cùng (robot_id, program_id) → tín hiệu gần như giống hệt
- **Flatline:** Kéo dài **40–60 steps** (trước đây: 20 steps) — rõ ràng hơn mức padding 0

## 5 loại lỗi được giữ lại
| Lỗi | Cảm biến | Cách phát hiện | Mức độ |
|-----|----------|----------------|--------|
| Spike (ngắn mạch) | S1 + S0 bình thường | Reconstruction error | Dễ |
| Thermal Drift (mất làm mát) | S2 tăng tuyến tính | So sánh baseline lịch sử | Trung bình |
| Flatline (mất tín hiệu) | S0 = 0 trong 40-60 steps | Local RevIN | Dễ |
| Cable Wear (mòn cáp) | S1 giảm gần 0, 8-15 lần | Reconstruction error | Dễ–Khó |
| Arc Blow (lệch từ) | S1 nhiễu ×3-5 | Cross-sensor context | Trung bình |

**Outputs:**
- `data/generated_sprint18b/train_welding_robot.csv` — 30 ngày, 4 robot, 100% normal (hội tụ)
- `data/generated_sprint18b/val_welding_robot.csv` — 7 ngày, 4 robot (validation)
- `data/generated_sprint18b/test_welding_robot.csv` — 2000 cycles, 50/50 normal/anomaly
- `data/generated_sprint18b/test_labels.csv` — nhãn anomaly theo cycle_id


In [ ]:
# %% Imports
from pathlib import Path
import sys

project_root = Path.cwd().parent.resolve()
print(f"Project root: {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.data_simulator.batch_generator import BatchGenerator
from backend.data_simulator.writer import CSVWriter
from backend.data_simulator.config import SimulatorConfig, PhysicsConfig, NoiseConfig

print("All imports OK")


## Step 1: Generate Training Data (30 days, 4 robots) — Convergent Mode

Sử dụng `BatchGenerator(training_mode=True)` để tạo dữ liệu huấn luyện hội tụ:
- **Sinh mẫu xác định:** Cùng (robot_id, program_id) → cùng envelope cơ sở
- **Giảm nhiễu ~50%:** Sử dụng `NoiseConfig.for_training()`
- **Kích thước cố định 512 steps** cho mọi cycle

Kết quả kỳ vọng: ~16.3 triệu rows, ~1.5 GB


In [ ]:
# %% Generate training data (convergent mode)
from datetime import datetime

# Sprint 18b: training_mode=True for convergent normal data
train_gen = BatchGenerator(seed=42, training_mode=True)
writer = CSVWriter(output_dir="F:/DataImpact/TED-Hackathon-Agent/data/generated_sprint18b")

print("Generating training data (30 days, 4 robots, convergent mode)...")
train_path, train_count = writer.write_train(
    train_gen,
    num_days=30,
    robots=[0, 1, 2, 3],
    start_date=datetime(2026, 3, 1),
)
print(f"Training data: {train_count:,} rows -> {train_path}")

# Separate validation set (different date range)
val_gen = BatchGenerator(seed=42, training_mode=True)
val_path, val_count = writer.write_train(
    val_gen,
    num_days=7,
    robots=[0, 1, 2, 3],
    start_date=datetime(2026, 4, 1),
    output_filename="val_welding_robot.csv",
)
print(f"Validation data: {val_count:,} rows -> {val_path}")


## Step 2: Generate Test Data (2000 cycles, 5 anomaly types)

Sử dụng `BatchGenerator()` (mặc định, không `training_mode`) để tạo dữ liệu test với:
- **5 loại lỗi:** spike, thermal_drift, flatline, cable_wear, arc_blow
- **50% anomalous cycles**
- **Nhiễu đầy đủ** (không giảm như training data)
- **Kích thước cố định 512 steps** cho mọi cycle


In [ ]:
# %% Generate test data with 5 anomaly types
test_gen = BatchGenerator(seed=42)

print("Generating test data (2000 cycles, 50% anomaly, 5 types)...")
test_path, labels_path, test_count = writer.write_test(
    test_gen,
    num_cycles=2000,
    anomaly_ratio=0.5,
    robots=[0, 1, 2, 3],
    seed=42,
    start_date=datetime(2026, 5, 1)
)
print(f"Test data: {test_count:,} rows -> {test_path}")
print(f"Labels: -> {labels_path}")


## Step 3: Verify Label Distribution

Kiểm tra: Chỉ 5 loại anomaly xuất hiện, phân bố labels, mỗi cycle = 512 steps.


In [ ]:
# %% Check label distribution, step count, anomaly types
import pandas as pd
import numpy as np

labels = pd.read_csv(labels_path)
test_df = pd.read_csv(test_path)

print("=== LABEL DISTRIBUTION ===")
print(labels["anomaly_type"].value_counts())
print(f"Normal cycles: {(labels['is_anomaly'] == 0).sum()}")
print(f"Anomalous cycles: {(labels['is_anomaly'] == 1).sum()}")

print("\n=== STEP COUNT PER CYCLE (should all be 512) ===")
steps = test_df.groupby("cycle_id").size()
print(f"Min: {steps.min()}, Max: {steps.max()}, Unique: {steps.nunique()}")
assert steps.min() == 512 and steps.max() == 512, "FAIL: step count != 512"
print("OK: All cycles have exactly 512 steps!")

# Verify only 5 anomaly types + 'normal'
anom_types = set(labels[labels["is_anomaly"] == 1]["anomaly_type"].unique())
print(f"\nAnomaly types found: {sorted(anom_types)} ({len(anom_types)} types)")
# Archived types should NOT appear
archived = {"phase_shift", "wire_stutter", "gas_blockage", "sticky_tc", "overshoot_degradation"}
found_archived = anom_types & archived
if found_archived:
    print(f"WARNING: Archived types found: {found_archived}")
else:
    print("OK: No archived anomaly types present")


## Step 4: Validate Flatline Duration

Kiểm tra flatline có độ dài **40-60 steps** (không còn bị nhầm với zero-padding).


In [ ]:
# %% Validate flatline duration
flat_cycles = labels[labels["anomaly_type"] == "flatline"]["cycle_id"].values[:10]
print("=== FLATLINE DURATION CHECK (expected 40-60 steps) ===")
for cid in flat_cycles:
    feed = test_df[test_df["cycle_id"] == cid]["wire_feed_speed_mm_min"].values
    zero = (feed < 0.1).astype(int)
    changes = np.diff(np.concatenate(([0], zero, [0])))
    runs = np.where(changes == -1)[0] - np.where(changes == 1)[0]
    longest = runs.max()
    ok = "OK" if 40 <= longest <= 60 else "FAIL"
    print(f"  Cycle {cid}: {longest} steps [{ok}]")


## Step 5: Validate Normal Data Convergence

Chứng minh dữ liệu train hội tụ: Cùng (robot_id, program_id) → tín hiệu gần như giống hệt.
- Overlay 3 cycles của cùng (robot=0, program=5)
- Sai số tối đa giữa các cycle < 5%


In [ ]:
# %% Validate convergence: overlay cycles of same (robot, program)
import matplotlib.pyplot as plt

train_df = pd.read_csv(train_path)
robot, prog = 0, 5
cycles = train_df[
    (train_df["robot_code"] == robot) & (train_df["program_number"] == prog)
]["cycle_id"].unique()[:3]

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
sensors = [
    ("wire_feed_speed_mm_min", "S0: Feed Speed (mm/min)"),
    ("welding_current_a", "S1: Welding Current (A)"),
    ("wire_temperature_c", "S2: Temperature (C)"),
]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for ax, (col, name) in zip(axes, sensors):
    for i, cid in enumerate(cycles):
        cycle = train_df[train_df["cycle_id"] == cid]
        ax.plot(cycle["serial_number"], cycle[col], color=colors[i],
                alpha=0.8, linewidth=0.7, label=f"Cycle {cid}")
    ax.set_ylabel(name, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[-1].set_xlabel("Serial Number (step)")
fig.suptitle(f"Data Convergence: Robot={robot}, Program={prog} (3 cycles overlaid)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Quantitative convergence check
base = train_df[train_df["cycle_id"] == cycles[0]].set_index("serial_number")
print("=== CONVERGENCE METRICS ===")
for col, name in [("wire_feed_speed_mm_min", "Feed"), ("welding_current_a", "Current"), ("wire_temperature_c", "Temp")]:
    max_dev = 0
    for cid in cycles[1:]:
        other = train_df[train_df["cycle_id"] == cid].set_index("serial_number")
        dev = np.abs(base[col].values - other[col].values).max()
        max_dev = max(max_dev, dev)
    pct = max_dev / base[col].max() * 100
    print(f"  {name}: max deviation = {max_dev:.2f} ({pct:.2f}%)")


## Step 6: Visualize Samples

Hiển thị 3-panel plot cho:
1. Một cycle normal (training data)
2. Một cycle mỗi loại anomaly (5 loại)

Chú thích: màu sắc phân biệt phase (idle/rampup/active/winddown/shutdown).


In [ ]:
# %% Visualize all sample types
SENSOR_COLS = ["wire_feed_speed_mm_min", "wire_temperature_c", "welding_current_a"]
SENSOR_NAMES = ["S0: Feed Speed (mm/min)", "S2: Temperature (C)", "S1: Welding Current (A)"]
PHASE_COLORS = {"idle": "#d3d3d3", "rampup": "#ffd700", "active": "#90ee90",
                "winddown": "#ffcc99", "shutdown": "#d3d3d3"}

# Merge labels into test_df for anomaly type
test_df = test_df.merge(labels[["cycle_id", "anomaly_type"]], on="cycle_id", how="left")

def plot_cycle(cycle_df, title=""):
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    for ax, col, name in zip(axes, SENSOR_COLS, SENSOR_NAMES):
        ax.plot(cycle_df["serial_number"], cycle_df[col], linewidth=0.8, color="#1f77b4")
        ax.set_ylabel(name, fontsize=9)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("Serial Number (step)")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Normal sample from training
print("=" * 60)
print("NORMAL SAMPLE (training data, convergent)")
print("=" * 60)
cid = train_df["cycle_id"].iloc[0]
cycle = train_df[train_df["cycle_id"] == cid]
meta = cycle.iloc[0]
plot_cycle(cycle, f"Normal (Train) Cycle #{cid} | Robot={meta['robot_code']} Program={meta['program_number']}")

# One of each anomaly type
print("=" * 60)
print("ANOMALY SAMPLES (5 types)")
print("=" * 60)
for anom_type in sorted(["spike", "thermal_drift", "flatline", "cable_wear", "arc_blow"]):
    subset = test_df[test_df["anomaly_type"] == anom_type]
    if len(subset) == 0:
        continue
    cid = subset["cycle_id"].iloc[0]
    cycle = test_df[test_df["cycle_id"] == cid]
    meta = cycle.iloc[0]
    plot_cycle(cycle, f"Anomaly: {anom_type.upper()} | Cycle #{cid} | Robot={meta['robot_code']} Program={meta['program_number']}")


## Step 7: Noise Level Comparison

So sánh mức nhiễu giữa training data (giảm ~50%) và test data (đầy đủ).
Chỉ tính residual std trong active phase (middle 60%) để tránh ảnh hưởng từ dynamics.


In [ ]:
# %% Noise level comparison
def active_residual_std(df, cid):
    vals = df[df["cycle_id"] == cid]["welding_current_a"].values
    s, e = int(len(vals) * 0.2), int(len(vals) * 0.8)
    return np.std(np.diff(vals[s:e])) / 1.414

# Training noise (reduced)
train_ids = train_df["cycle_id"].unique()[:50]
train_noises = [active_residual_std(train_df, c) for c in train_ids]
# Test noise (full)
norm_ids = labels[labels["anomaly_type"] == "normal"]["cycle_id"].values[:50]
norm_noises = [active_residual_std(test_df, c) for c in norm_ids]

print("=== NOISE LEVEL COMPARISON ===")
print(f"Train (reduced noise): mean={np.mean(train_noises):.4f} A, std={np.std(train_noises):.4f}")
print(f"Test  (full noise):    mean={np.mean(norm_noises):.4f} A, std={np.std(norm_noises):.4f}")
ratio = np.mean(norm_noises) / max(np.mean(train_noises), 0.0001)
print(f"Ratio test/train: {ratio:.2f}x (expected > 1.5x)")

# Bar chart
plt.figure(figsize=(8, 4))
plt.bar(["Train (reduced noise)", "Test (full noise)"],
        [np.mean(train_noises), np.mean(norm_noises)],
        yerr=[np.std(train_noises), np.std(norm_noises)],
        capsize=5, color=["#2ca02c", "#d62728"], alpha=0.7)
plt.ylabel("Residual Std Dev (A)")
plt.title("Noise Level: Train vs Test (active phase)")
plt.grid(True, alpha=0.3, axis="y")
plt.show()


## Step 8: Training Instructions

Sau khi generate dữ liệu, train model GLCReconV8 với:

```python
from backend.data_simulator.batch_generator import BatchGenerator
from backend.data_simulator.writer import CSVWriter

# Training data -> convergent mode
train_gen = BatchGenerator(seed=42, training_mode=True)
writer = CSVWriter(output_dir="data/generated_sprint18b")
writer.write_train(train_gen, num_days=30, robots=[0,1,2,3],
                   start_date=datetime(2026, 3, 1))

# Validation data -> convergent mode
val_gen = BatchGenerator(seed=42, training_mode=True)
writer.write_train(val_gen, num_days=7, robots=[0,1,2,3],
                   start_date=datetime(2026, 4, 1),
                   output_filename="val_welding_robot.csv")

# Test data -> normal mode (full noise)
test_gen = BatchGenerator(seed=42)
writer.write_test(test_gen, num_cycles=2000, anomaly_ratio=0.5,
                  robots=[0,1,2,3], seed=42,
                  start_date=datetime(2026, 5, 1))
```

**Lưu ý:** Dữ liệu train đã được generate với `training_mode=True`:
- Noise giảm ~50% → model học signal structure dễ hơn
- Deterministic seeding → cùng (robot, program) cho cùng base envelope
- Tất cả cycle = 512 steps → không cần padding 0

### Tham khảo
- `GENERATION-GUIDE.md` — đặc tả kỹ thuật đầy đủ
- `welding-program-research.md` — research về thông số hàn thực tế
